In [ ]:
!nvidia-smi

Sat Oct 31 16:07:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.21.0
Uninstalling transformers-5.21.0:
  Successfully uninstalled transformers-5.21.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]


In [ ]:
!pip install evaluate -q


In [ ]:
!pip install rouge_score -q


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
from evaluate import load
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.28GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip

--2026-10-31 16:09:18--  https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip [following]
--2026-10-31 16:09:18--  https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: ‘summarizer-data.zip’

summarizer-data.zip 100%[===================>]   7.54M  25.8MB/s    in 0.3s    

2026-10-31 16:09:19 (25.8 MB/s) - ‘summarizer-data.zip’ saved [7903594

In [ ]:
from datasets import load_from_disk
dataset_samsum  = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [ ]:
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]
print(f"Split lengths:{split_lengths}")
print(f"Features:{dataset_samsum['train'].column_names}")
print("\nDialogue:")
print(dataset_samsum["test"][2]["dialogue"])
print("\nSummary")

print(dataset_samsum["test"][2]["summary"])

Split lengths:[14732, 819, 818]
Features:['id', 'dialogue', 'summary']

Dialogue:
Lenny: Babe, can you help me with something?
Bob: Sure, what's up?
Lenny: Which one should I pick?
Bob: Send me photos
Lenny:  <file_photo>
Lenny:  <file_photo>
Lenny:  <file_photo>
Bob: I like the first ones best
Lenny: But I already have purple trousers. Does it make sense to have two pairs?
Bob: I have four black pairs :D :D
Lenny: yeah, but shouldn't I pick a different color?
Bob: what matters is what you'll give you the most outfit options
Lenny: So I guess I'll buy the first or the third pair then
Bob: Pick the best quality then
Lenny: ur right, thx
Bob: no prob :)

Summary
Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.


In [ ]:
def convert_examples_to_features(example_batch):
    # Tokenize inputs and targets together using the modern API
    model_inputs = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True
    )

    # Tokenize the target summaries using text_target
    labels = tokenizer(
        text_target=example_batch['summary'],
        max_length=128,
        truncation=True
    )

    # Add the target IDs into the dictionary as 'labels'
    model_inputs['labels'] = labels['input_ids']

    return model_inputs


In [ ]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features,batched=True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [ ]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

trainer_args = Seq2SeqTrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',          # Fixed: Changed evaluation_strategy -> eval_strategy
    eval_steps=500,
    save_steps=1_000_000,           # Cleaner integer notation instead of float 1e6
    gradient_accumulation_steps=16,
    predict_with_generate=True      # Enables decoding summaries for metric scoring
)


In [ ]:
trainer = Trainer(
    model=model_pegasus,args =trainer_args,
    processing_class=tokenizer,data_collator=seq2seq_data_collator,
    train_dataset = dataset_samsum_pt["train"],
    eval_dataset = dataset_samsum_pt["validation"]
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
500,27.142426,1.495711
921,25.976559,1.442153


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=921, training_loss=29.777252259394245, metrics={'train_runtime': 2851.3999, 'train_samples_per_second': 5.167, 'train_steps_per_second': 0.323, 'total_flos': 5535530955915264.0, 'train_loss': 29.777252259394245, 'epoch': 1.0})

In [ ]:
from tqdm import tqdm  # Explicitly import the tqdm class
import torch

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu",
                                column_text="dialogue",
                                column_summary="summary"):

    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    # Fixed: Changed tqdm.tqdm to just tqdm to match the explicit import above
    for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                           padding="max_length", return_tensors="pt")

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=4,
            max_length=128
        )

        decoded_summaries = tokenizer.batch_decode(summaries, skip_special_tokens=True)
        decoded_summaries = [d.strip() for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    score = metric.compute()
    return score


In [ ]:
import random

# 1. Pick a random sample from the test split
test_sample = dataset_samsum["test"][random.randint(0, len(dataset_samsum["test"]) - 1)]

print("💬 --- ORIGINAL DIALOGUE ---")
print(test_sample["dialogue"])
print("\n🎯 --- REFERENCE SUMMARY (HUMAN) ---")
print(test_sample["summary"])

# 2. Tokenize the input dialogue text
inputs = tokenizer(test_sample["dialogue"], return_tensors="pt", max_length=1024, truncation=True)

# 3. Move tensors to GPU if available
device = model_pegasus.device
inputs = {k: v.to(device) for k, v in inputs.items()}

# 4. Generate the summary tokens using the model
generated_tokens = model_pegasus.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=128,
    num_beams=4,           # Beam search for higher quality text generation
    length_penalty=0.8,
    early_stopping=True
)

# 5. Decode tokens back to readable text
decoded_summary = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

print("\n🤖 --- MODEL GENERATED SUMMARY ---")
print(decoded_summary)


💬 --- ORIGINAL DIALOGUE ---
Lincoln: Heeyyy ;* whats up
Fatima: I talked to Jenson, he’s not too happy ;p
Lincoln: the place sucks??
Fatima: No, the place is ok, I think, we can go there, it’s about Alene
Lincoln: typical, dont worry about it
Fatima: He thinks she may have a depression :[
Lincoln: nothin new, everyone has it, she needs a doctor then
Fatima: But she won’t go ;/
Lincoln: so she’s destroying her life fuck it its not your problem
Fatima: It is, they’re both my friends!
Lincoln: you better think what to do if they break up
Fatima: Ehh yes Ill have a problem ;//
Lincoln: both blaming each other and talking with you about it, perfect
Fatima: Alene is just troubled… She’d been through a lot…
Lincoln: everyone has their problems, the question is are ya doin sth about them
Fatima: She has problems facing it, don’t be surprised :[
Lincoln: then it is her problem
Fatima: You are so cruel at times… o.O
Lincoln: maybe, for me its just a common sense
Fatima: Why can’t everyone be jus

In [ ]:

rouge_names = ["rouge1","rouge2","rougeL","rougeLsum"]
rouge_metric = load('rouge')

In [ ]:

# 1. Cleaned up typos ('dialogue' spelling, model reference)
score = calculate_metric_on_test_ds(
    dataset=dataset_samsum['test'],
    metric=rouge_metric,
    model=model_pegasus,            # Using your declared model object
    tokenizer=tokenizer,
    batch_size=2,
    column_text='dialogue',         # Fixed: 'dialouge' -> 'dialogue'
    column_summary='summary'
)

# 2. Modern evaluate parsing: scores are now raw floats, no '.mid.fmeasure' needed!
rouge_dict = {rn: score[rn] for rn in rouge_names}

# 3. Create and display the DataFrame
df_results = pd.DataFrame(rouge_dict, index=['pegasus'])
df_results


100%|██████████| 410/410 [15:06<00:00,  2.21s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.442945,0.212042,0.350259,0.350273


In [ ]:
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')